In [ ]:
from statistics import median

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings

from sklearn.calibration import calibration_curve

warnings.filterwarnings('ignore')

## HW14 — многоклассовая классификация OvR/OvO

По Sigma Cabs предскажите один из трёх типов повышенного тарифа. Это multiclass с одной меткой на поездку, не multilabel. Полные данные не пересчитывались.

In [ ]:
df = pd.read_csv('./sigma_cabs.csv')
df.shape

In [ ]:
# Занесем индекс колонку
df = df.set_index('Trip_ID')
df.head()

### Признаки

Trip_ID идентифицирует поездку. Trip_Distance — расстояние; TypeofCab — тип машины; CustomerSinceMonths — стаж клиента; LifeStyleIndex и ConfidenceLifeStyle_Index — показатели клиента; Destination_Type — категория назначения; Customer_Rating — рейтинг; CancellationLast1Month — отмены; Var1/Var2/Var3 — скрытые числовые показатели; Gender — пол. Таргет — Surge_Pricing_Type.

### Подготовка

Заполните числовые пропуски медианами, категориальные модами. Постройте корреляционную матрицу и выведите пять наиболее коррелированных пар. Для категорий используйте OHE с drop_first=True.

In [ ]:
### Your code is here

number_values = df.select_dtypes(include='number')
categorical_values = df.select_dtypes(include='object')

for col in number_values:
    df[col] = df[col].fillna(df[col].median())

In [ ]:

# 1. Заполняем пропуски ДО кодирования (самым частым значением / модой)
for col in categorical_values.columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# 2. Применяем get_dummies к нужным колонкам и ОБЪЕДИНЯЕМ с исходным датафреймом
df = pd.get_dummies(data=df, columns=categorical_values.columns, drop_first=True)


In [ ]:
fig = plt.figure()
fig.set_size_inches(20, 10)
sns.heatmap(df.corr(), annot=True)

In [ ]:
### Секретные функции для фильтрации признаков

def get_redundant_pairs(df):
    pairs_to_drop = set()
    cols = df.columns
    for i in range(0, df.shape[1]):
        for j in range(0, i+1):
            pairs_to_drop.add((cols[i], cols[j]))
    return pairs_to_drop

def get_top_abs_correlations(df, n=5):
    au_corr = df.corr().abs().unstack()
    labels_to_drop = get_redundant_pairs(df)
    au_corr = au_corr.drop(labels=labels_to_drop).sort_values(ascending=False)
    return au_corr[0:n]

print("Top Absolute Correlations")
print(get_top_abs_correlations(df[number_values.columns], 5))

In [ ]:
df.info()

### Обучение

In [ ]:
np.random.seed(2022)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:

X = df.drop('Surge_Pricing_Type', axis=1)
y = df['Surge_Pricing_Type']
X_train, X_test, y_train, y_test  = train_test_split(X, y,
                                                     test_size=0.2,
                                                     shuffle=True,
                                                     random_state=2022)

### Задание 1: OvR

Обучите OneVsRestClassifier с LogisticRegression и StandardScaler. Посчитайте precision, recall и F1 с micro, macro и weighted усреднением; вывод округлите до трёх знаков.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

### Your code is here

pipe = Pipeline([('scaler', StandardScaler()),
                 ('one_vs_rest',OneVsRestClassifier(LogisticRegression()))])


pipe.fit(X_train, y_train)

pipe.score(X_test, y_test)

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_test, pipe.predict(X_test),digits=3)
print(report)

Подберите предложенные гиперпараметры GridSearchCV по accuracy. Fit выполняйте на train; используйте стандартную пятикратную CV и итоговые метрики на test.

In [ ]:
param_grid = {'one_vs_rest__estimator__solver': ['saga'],
              'one_vs_rest__estimator__l1_ratio': [0.5],
              'one_vs_rest__estimator__penalty': ['l1', 'l2', 'elasticnet'],
              'one_vs_rest__estimator__C': [0.001, 0.01, 0.1, 1]}

In [ ]:
### Your code is here

from sklearn.model_selection import GridSearchCV


search = GridSearchCV(estimator=pipe,param_grid=param_grid)

search.fit(X_train, y_train)

print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)


report_2 = classification_report(y_test, search.predict(X_test), digits=3)
print(report_2)

Постройте калибровочные кривые для каждого класса против остальных. Сопоставляйте столбцы predict_proba с classes_.

In [ ]:
### Your code is here
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# Вероятности всех классов
y_proba = search.predict_proba(X_test)

# Реальные метки классов
classes = search.classes_

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, ax in enumerate(axes):
    # Превращаем задачу в бинарную: класс против остальных
    y_binary = (np.asarray(y_test) == classes[i]).astype(int)

    # Вероятности текущего класса
    prob = y_proba[:, i]

    # Вычисляем точки калибровочной кривой
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_binary,
        prob,
        n_bins=10,
        strategy="quantile"
    )

    ax.plot(
        mean_predicted_value,
        fraction_of_positives,
        marker="o",
        label=f"Class {classes[i]}"
    )

    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")

    ax.set_title(f"{i}-vs-rest (class {classes[i]})")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


### Задание 2: степени признаков

Для базовых вещественных колонок добавьте степени до четвёртой, без перемножения разных признаков и без возведения бинарных колонок в степени. Обучите логистическую регрессию с гиперпараметрами лучшей модели первого задания и сравните метрики.

In [ ]:
### Создание полиномиальных признаков
X_polinomial = X.copy()

# Исключаем целевой признак из списка, чтобы не было KeyError
number_values = [col for col in number_values if col != 'Surge_Pricing_Type']

### Your code is here
for col in number_values:
    for power in [1,2,3,4]:
        to_add = (X_polinomial[col]**power).to_frame().rename({col:f"{col}_{power}"}, axis=1)
        X_polinomial = pd.concat((X_polinomial, to_add), axis=1)


In [ ]:
X_polinomial

In [ ]:
X_pol_train, X_pol_test, y_train, y_test  = train_test_split(X_polinomial, y,
                                                             test_size=0.2,
                                                             shuffle=True,
                                                             random_state=2022)

In [ ]:
### Your code is here
search = GridSearchCV(estimator=pipe,param_grid={key: [value] for key, value in search.best_params_.items()})

search.fit(X_pol_train, y_train)

print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)


report_3 = classification_report(y_test, search.predict(X_pol_test), digits=3)
print(report_3)


Повторите калибровочные кривые для модели со степенями признаков и сравните результат.

In [ ]:
### Your code is here
import numpy as np
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# Вероятности всех классов
y_proba = search.predict_proba(X_pol_test)

# Реальные метки классов
classes = search.classes_

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, ax in enumerate(axes):
    # Превращаем задачу в бинарную: класс против остальных
    y_binary = (np.asarray(y_test) == classes[i]).astype(int)

    # Вероятности текущего класса
    prob = y_proba[:, i]

    # Вычисляем точки калибровочной кривой
    fraction_of_positives, mean_predicted_value = calibration_curve(
        y_binary,
        prob,
        n_bins=10,
        strategy="quantile"
    )

    ax.plot(
        mean_predicted_value,
        fraction_of_positives,
        marker="o",
        label=f"Class {classes[i]}"
    )

    ax.plot([0, 1], [0, 1], "--", color="gray", label="Perfect calibration")

    ax.set_title(f"{i}-vs-rest (class {classes[i]})")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()



### Задание 3: OvO

На исходных признаках обучите OneVsOneClassifier с SGDClassifier. hinge соответствует SVM, log_loss — логистической функции потерь. Выведите precision, recall, F1 и их усреднения.

In [ ]:
X_train, X_test, y_train, y_test  = train_test_split(X, y,
                                                     test_size=0.2,
                                                     shuffle=True,
                                                     random_state=2022)

In [ ]:
from sklearn.linear_model import SGDClassifier
from sklearn.multiclass import OneVsOneClassifier

### Your code is here
pipe_all_all = Pipeline([('scaler', StandardScaler()),
                         ('all_vs_all',OneVsOneClassifier(SGDClassifier()))])

pipe_all_all.fit(X_train, y_train)

report_4 = classification_report(y_test, pipe_all_all.predict(X_test), digits=3)
print(report_4)

In [ ]:
# pipe_all_all.get_params()

Подберите функцию потерь, penalty и alpha с помощью CV на train; итоговые метрики посчитайте на test.

In [ ]:
param_grid = {'all_vs_all__estimator__loss': ['hinge', 'log_loss', 'modified_huber'],
              'all_vs_all__estimator__penalty': ['l1', 'l2'],
              'all_vs_all__estimator__alpha': [0.001, 0.01, 0.1]}

In [ ]:
### Your code is here
from sklearn.model_selection import PredefinedSplit

# test_fold = [0 if x in X_train.index else -1 for x in X.index]
# ps = PredefinedSplit(test_fold)

search = GridSearchCV(estimator=pipe_all_all,param_grid=param_grid)

search.fit(X_train, y_train)

print(f"Best parameter (CV score={search.best_score_:.5f}):")
print(search.best_params_)

report_5 = classification_report(y_test, search.predict(X_test), digits=3)
print(report_5)

In [ ]:
report_5 = classification_report(y_test, search.predict(X_test), digits=3)
print(report_5)

Можно ли по результатам однозначно предпочесть OvR или OvO? Собственный письменный вывод не заполнен.